# 11.2 — Markov Decision Processes

A Markov Decision Process (MDP) is the smallest bookkeeping system that makes sequential choice computable: states say where the agent is, actions say what it can do, transition probabilities say where each action may lead, rewards say what each jump pays, and discounting says how strongly later consequences count. In this lesson, you will build those objects from scratch with NumPy, then use them to compute returns, Bellman backups, value updates, policies, and exploration bonuses.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build Markov Decision Processes one idea at a time. Run each cell in order and read the printed intermediate values — every state, action, probability, reward, discount, and target is made visible so the formulas never feel like black boxes. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, transition tensors, and expectation arithmetic.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for any random choices.

### 1. States, actions, transitions, and rewards as tensors

An MDP starts by naming **states** and **actions**, then writing a transition table $P(s'\mid s,a)$ and a reward table $R(s,a,s')$. The transition row for a fixed state-action pair must sum to 1 because it is a probability distribution over next states. The reward table attaches a consequence to each possible jump, not just to the current state, because the same action can pay differently depending on where it lands.

In [ ]:
states_w = ["start", "study", "rest"]  # three possible situations for the agent.
actions_w = ["focus", "break"]  # two choices available in each state.
S_w, A_w = len(states_w), len(actions_w)  # state and action counts.
P_w = np.zeros((S_w, A_w, S_w))  # P[s, a, s_next] stores transition probabilities.
R_w = np.zeros((S_w, A_w, S_w))  # R[s, a, s_next] stores rewards for realized transitions.
print("P shape:", P_w.shape, "R shape:", R_w.shape)

▶ What you'll see: both tensors have shape `(3, 2, 3)`, meaning each state-action pair has three possible next states.

In [ ]:
P_w[0, 0] = [0.10, 0.80, 0.10]  # from start, focus usually reaches study.
P_w[0, 1] = [0.20, 0.20, 0.60]  # from start, break usually reaches rest.
P_w[1, 0] = [0.00, 0.70, 0.30]  # from study, focus often keeps studying.
P_w[1, 1] = [0.10, 0.20, 0.70]  # from study, break often rests.
P_w[2, 0] = [0.10, 0.60, 0.30]  # from rest, focus can return to study.
P_w[2, 1] = [0.20, 0.10, 0.70]  # from rest, break mostly stays resting.
R_w[:, :, :] = [[[-0.2, 2.0, 0.5], [0.0, 0.4, 0.8]],
                [[0.0, 1.0, 0.2], [0.1, 0.3, 1.0]],
                [[0.0, 1.5, 0.6], [0.2, 0.1, 0.7]]]
print("row sums for P:\n", P_w.sum(axis=2))
assert np.allclose(P_w.sum(axis=2), 1.0)

▶ What you'll see: every state-action row sums to 1, so each row is a valid distribution over next states.

In [ ]:
plt.figure(figsize=(5, 3))
plt.imshow(P_w[0], cmap="viridis", aspect="auto")
plt.colorbar(label="probability")
plt.xticks(range(S_w), states_w); plt.yticks(range(A_w), actions_w)
plt.title("1: next-state probabilities from start")
plt.show()

▶ What you'll see: `focus` puts most probability on `study`, while `break` puts most probability on `rest`.

*Why it's done this way:* the Markov assumption says the current state contains the needed history, so $P[s,a,s']$ is enough to average over the future. Separating $P$ from $R$ matters: probability says what can happen, reward says how much each realized consequence is worth.

### 2. Discounted return turns a reward stream into one target

A reward stream must be collapsed into one number before it can train a value. The discounted return is $G=r_0+\gamma r_1+\gamma^2 r_2+\cdots$, where $0\le\gamma<1$. Discounting makes delayed reward count, but count less than immediate reward, which is how RL can value durable plans without pretending the future is free.

In [ ]:
rewards_w = np.array([1.0, 0.0, 2.0])  # immediate reward, then two future rewards.
gamma_w = 0.9  # discount factor.
powers_w = gamma_w ** np.arange(len(rewards_w))  # [1, gamma, gamma^2].
print("discount powers:", np.round(powers_w, 3))
print("discounted pieces:", np.round(powers_w * rewards_w, 3))

▶ What you'll see: the delayed reward `2` is multiplied by `0.9²`, so it contributes `1.62`.

In [ ]:
G_w = float(np.sum(powers_w * rewards_w))
print("three-step return:", round(G_w, 3))
assert round(G_w, 3) == 2.620
plt.figure(figsize=(4.5, 3))
plt.bar(["r0", "γ r1", "γ² r2"], powers_w * rewards_w, color="teal")
plt.title("2: discounted return pieces")
plt.ylabel("contribution to G")
plt.show()

▶ What you'll see: the return is `1 + 0 + 1.620 = 2.620`, with the late reward visibly discounted.

*Why it's done this way:* the powers of $\gamma$ encode time preference and keep infinite sums finite. If $\gamma$ is high, the agent plans far ahead; if it is low, the agent behaves more myopically.

### 3. Bellman expectation backup averages over policy and dynamics

A value $V^\pi(s)$ is the expected discounted consequence of starting in state $s$ and following policy $\pi$. The Bellman expectation equation expands that statement one step: average over the action sampled by the policy, then average over the next state sampled by the transition model, and add reward plus discounted next value.

In [ ]:
pi_w = np.array([[0.75, 0.25],  # in start, usually focus.
                 [0.65, 0.35],  # in study, mostly focus.
                 [0.55, 0.45]]) # in rest, slight preference for focus.
V_old_w = np.array([0.0, 1.2, 0.7])  # current guesses for state values.
print("policy row sums:", pi_w.sum(axis=1))
assert np.allclose(pi_w.sum(axis=1), 1.0)

▶ What you'll see: each state has a probability distribution over the two actions.

In [ ]:
s_w = 0  # back up the value of start.
action_terms_w = []
for a_w in range(A_w):
    next_terms_w = P_w[s_w, a_w] * (R_w[s_w, a_w] + gamma_w * V_old_w)
    action_terms_w.append(float(np.sum(next_terms_w)))
    print(actions_w[a_w], "next-state terms:", np.round(next_terms_w, 3), "sum:", round(action_terms_w[-1], 3))

▶ What you'll see: each action has its own expected reward-plus-future value, computed across next states.

In [ ]:
V_start_w = float(np.dot(pi_w[s_w], action_terms_w))
print("Bellman backup for V(start):", round(V_start_w, 3))
assert round(V_start_w, 3) == 2.206
plt.figure(figsize=(4.5, 3))
plt.bar(actions_w, action_terms_w, color="seagreen")
plt.title("3: action consequences before policy weighting")
plt.ylabel("expected r + γV(s')")
plt.show()

▶ What you'll see: the state value is a policy-weighted average of the two action consequences.

*Why it's done this way:* the nested sums are just expectation bookkeeping. The policy decides how often each action is tried; the transition probabilities decide how often each next state occurs; the value estimate turns one-step consequences into long-run consequences.

### 4. Action values and one-step bootstrap updates

A state value averages over actions, but an action value $Q(s,a)$ asks what happens if we choose a particular action now and follow a plan later. A one-step bootstrap target uses one observed reward plus the current estimate of the next state: $y=r+\gamma V(s')$. The update $Q\leftarrow Q+\alpha(y-Q)$ moves partway toward that target.

In [ ]:
r_obs_w = 1.0  # observed immediate reward.
next_value_w = 0.8  # current estimate for V(s').
q_old_w = 0.4  # current estimate for Q(s,a).
alpha_w = 0.5  # learning rate: move halfway toward the target.
target_w = r_obs_w + gamma_w * next_value_w
print("bootstrap target:", round(target_w, 3))
assert round(target_w, 3) == 1.720

▶ What you'll see: the target is `1 + 0.9·0.8 = 1.720`.

In [ ]:
q_new_w = q_old_w + alpha_w * (target_w - q_old_w)
print("Q old:", q_old_w, "Q new:", round(q_new_w, 3))
assert round(q_new_w, 3) == 1.060
plt.figure(figsize=(4.5, 3))
plt.bar(["old Q", "target", "new Q"], [q_old_w, target_w, q_new_w], color=["gray", "black", "teal"])
plt.title("4: one-step bootstrapped update")
plt.ylabel("value")
plt.show()

▶ What you'll see: the new estimate lands halfway between the old estimate and the target.

*Why it's done this way:* bootstrapping lowers variance by not waiting for a complete episode, but it introduces bias because the target uses the learner's own estimate. The learning rate controls how much one noisy transition can overwrite the table.

### 5. Policies turn scores into action probabilities

A policy can be represented by logits, then converted to probabilities with softmax. The exponentials make larger logits receive more probability while still assigning every action nonzero support. Once we have probabilities, expected immediate reward is just a weighted average over actions.

In [ ]:
logits_w = np.array([1.0, 0.0])  # action preferences before normalization.
exp_w = np.exp(logits_w - np.max(logits_w))  # stable exponentials.
probs_w = exp_w / exp_w.sum()
print("softmax probabilities:", np.round(probs_w, 3))
assert np.allclose(np.round(probs_w, 3), [0.731, 0.269])

▶ What you'll see: action 0 receives about `0.731` probability and action 1 receives about `0.269`.

In [ ]:
rewards_actions_w = np.array([2.0, 0.0])
expected_reward_w = float(np.dot(probs_w, rewards_actions_w))
print("expected reward:", round(expected_reward_w, 3))
assert round(expected_reward_w, 3) == 1.462
plt.figure(figsize=(4.5, 3))
plt.bar(actions_w, probs_w, color="darkorange")
plt.title("5: softmax policy probabilities")
plt.ylabel("π(a|s)")
plt.show()

▶ What you'll see: most probability mass is on `focus`, so the expected reward is close to but below 2.

*Why it's done this way:* logits are unconstrained numbers that optimization can change freely; softmax converts them into a legal probability distribution. Moving probability mass changes expected consequence, which is the bridge from value estimates to policy improvement.

### 6. Exploration bonuses pay for useful uncertainty

If an action has been tried only a few times, its sample mean may be misleading. A UCB-style index adds an uncertainty bonus to the mean: $\bar r + c\sqrt{2\ln(t)/N}$. The bonus shrinks as the action count $N$ grows, so curiosity is temporary rather than permanent.

In [ ]:
mean_reward_w = 0.55
count_w = 5
time_w = 20
c_w = 1.0
bonus_w = c_w * np.sqrt(2 * np.log(time_w) / count_w)
ucb_w = mean_reward_w + bonus_w
print("bonus:", round(bonus_w, 3), "UCB index:", round(ucb_w, 3))
assert round(ucb_w, 3) == 1.645

▶ What you'll see: the index is much larger than the observed mean because the arm is still uncertain.

In [ ]:
counts_w = np.arange(1, 31)
bonuses_w = np.sqrt(2 * np.log(time_w) / counts_w)
plt.figure(figsize=(5, 3))
plt.plot(counts_w, mean_reward_w + bonuses_w, color="crimson")
plt.axhline(mean_reward_w, color="gray", linestyle="--", label="mean reward")
plt.title("6: UCB bonus shrinks with count")
plt.xlabel("times action was tried")
plt.ylabel("mean + uncertainty bonus")
plt.legend()
plt.show()

▶ What you'll see: the exploration index falls toward the empirical mean as the action receives more samples.

*Why it's done this way:* greedy reward chasing can get stuck because it never gathers missing evidence. An exploration bonus is a principled optimism term: uncertain actions are temporarily treated as potentially good until data proves otherwise.


## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per computational mechanic in this lesson. Each uses a few
> small numbers, prints the intermediate values with inline `# ->` comments, draws one picture, and
> ends with an `assert` that pins the answer. Run them top to bottom.

### ✍️ Toy 1 · Transition rows and rewards align

For one state-action pair, the transition row is a probability distribution and the reward row says
what each possible landing pays.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_rng = np.random.default_rng(0)
t1_P = np.array([[[1.0, 0.0], [0.25, 0.75]], [[0.5, 0.5], [0.0, 1.0]]])
t1_R = np.array([[[0.0, 0.0], [0.0, 4.0]], [[1.0, 0.0], [0.0, 2.0]]])
t1_state = 0
t1_action = 1
t1_probs = t1_P[t1_state, t1_action]             # -> [0.25, 0.75]
t1_rewards = t1_R[t1_state, t1_action]           # -> [0.0, 4.0]
t1_row_sum = float(np.sum(t1_probs))             # -> 1.0
t1_weighted = t1_probs * t1_rewards              # -> [0.0, 3.0]
t1_expected_reward = float(np.sum(t1_weighted))  # -> 3.0
print("P shape:", t1_P.shape)                    # -> (2, 2, 2)
print("transition row:", t1_probs.tolist())      # -> [0.25, 0.75]
print("row sum:", round(t1_row_sum, 3))          # -> 1.0
print("reward row:", t1_rewards.tolist())        # -> [0.0, 4.0]
print("weighted rewards:", t1_weighted.tolist()) # -> [0.0, 3.0]
print("expected reward:", t1_expected_reward)    # -> 3.0

plt.figure(figsize=(4.3, 2.6))
plt.bar(["s'0", "s'1"], t1_probs, color="seagreen")
plt.ylim(0, 1)
plt.title("Toy 1 · P(s' | s=0,a=1)")
plt.ylabel("probability")
plt.show()
assert t1_row_sum == 1.0 and t1_expected_reward == 3.0

▶ What you'll see: next state `1` is more likely and carries all of the expected reward.

### ✍️ Toy 2 · Discounted return is a single target

A reward stream becomes one target by multiplying each reward by the right power of `γ`.

In [ ]:
import numpy as np

t2_rng = np.random.default_rng(0)
t2_rewards = np.array([2.0, 0.0, 3.0])
t2_gamma = 0.5
t2_discounts = t2_gamma ** np.arange(t2_rewards.size)      # -> [1.0, 0.5, 0.25]
t2_terms = t2_rewards * t2_discounts                       # -> [2.0, 0.0, 0.75]
t2_return = float(np.sum(t2_terms))                        # -> 2.75
print("rewards:", t2_rewards.tolist())                    # -> [2.0, 0.0, 3.0]
print("discounts:", t2_discounts.tolist())                # -> [1.0, 0.5, 0.25]
print("terms:", t2_terms.tolist())                        # -> [2.0, 0.0, 0.75]
print("return:", t2_return)                               # -> 2.75

plt.figure(figsize=(4.3, 2.6))
plt.bar(["t0", "t1", "t2"], t2_terms, color="teal")
plt.title("Toy 2 · discounted return pieces")
plt.ylabel("γ^t r_t")
plt.show()
assert t2_return == 2.75

▶ What you'll see: the final reward `3` contributes `0.75` after two discounts.

### ✍️ Toy 3 · Bellman expectation nests two averages

A policy value first averages each action over possible next states, then averages those action values
under the policy probabilities.

In [ ]:
import numpy as np

t3_rng = np.random.default_rng(0)
t3_P = np.array([[[0.5, 0.5], [0.2, 0.8]], [[0.0, 1.0], [1.0, 0.0]]])
t3_R = np.array([[[0.0, 2.0], [1.0, 0.0]], [[0.0, 1.0], [2.0, 0.0]]])
t3_V = np.array([1.0, 3.0])
t3_pi = np.array([0.25, 0.75])
t3_gamma = 0.5
t3_terms_a0 = t3_P[0, 0] * (t3_R[0, 0] + t3_gamma * t3_V)  # -> [0.25, 1.75]
t3_q_a0 = float(np.sum(t3_terms_a0))                       # -> 2.0
t3_terms_a1 = t3_P[0, 1] * (t3_R[0, 1] + t3_gamma * t3_V)  # -> [0.3, 1.2]
t3_q_a1 = float(np.sum(t3_terms_a1))                       # -> 1.5
t3_action_values = np.array([t3_q_a0, t3_q_a1])            # -> [2.0, 1.5]
t3_backup = float(t3_pi @ t3_action_values)                # -> 1.625
print("a0 next-state terms:", t3_terms_a0.tolist())        # -> [0.25, 1.75]
print("a1 next-state terms:", np.round(t3_terms_a1, 3).tolist())  # -> [0.3, 1.2]
print("action values:", np.round(t3_action_values, 3).tolist())  # -> [2.0, 1.5]
print("policy weights:", t3_pi.tolist())                  # -> [0.25, 0.75]
print("Bellman backup:", round(t3_backup, 3))             # -> 1.625

plt.figure(figsize=(4.3, 2.6))
plt.bar(["a0", "a1"], t3_action_values, color="steelblue")
plt.axhline(t3_backup, color="black", linestyle="--", label="policy average")
plt.title("Toy 3 · action values before policy weighting")
plt.ylabel("expected r + γV")
plt.legend()
plt.show()
assert round(float(t3_backup), 3) == 1.625

▶ What you'll see: the policy leans toward action `1`, so the backup is closer to `1.5` than to `2.0`.

### ✍️ Toy 4 · One sampled transition updates Q

A sampled transition creates one bootstrap target for one table cell, then the learning rate applies a
partial correction.

In [ ]:
import numpy as np

t4_rng = np.random.default_rng(0)
t4_V = np.array([1.0, 0.0])
t4_reward = 2.0
t4_gamma = 0.75
t4_next_state = 0
t4_q_old = 1.25
t4_alpha = 0.4
t4_future = t4_gamma * t4_V[t4_next_state]       # -> 0.75
t4_target = t4_reward + t4_future                # -> 2.75
t4_error = t4_target - t4_q_old                  # -> 1.5
t4_step = t4_alpha * t4_error                    # -> 0.6
t4_q_new = t4_q_old + t4_step                    # -> 1.85
print("next-state values:", t4_V.tolist())       # -> [1.0, 0.0]
print("target:", round(float(t4_target), 3))     # -> 2.75
print("TD error:", round(float(t4_error), 3))    # -> 1.5
print("update step:", round(float(t4_step), 3))  # -> 0.6
print("new Q:", round(float(t4_q_new), 3))       # -> 1.85

plt.figure(figsize=(4.3, 2.6))
plt.bar(["old Q", "target", "new Q"], [t4_q_old, t4_target, t4_q_new], color=["gray", "black", "teal"])
plt.title("Toy 4 · bootstrapped Q update")
plt.ylabel("value")
plt.show()
assert round(float(t4_q_new), 3) == 1.85

▶ What you'll see: the old estimate moves `0.6` toward the target, not all the way to it.

### ✍️ Toy 5 · Softmax makes logits legal probabilities

Softmax preserves the ranking of action scores while normalizing them to sum to one.

In [ ]:
import numpy as np

t5_rng = np.random.default_rng(0)
t5_logits = np.array([0.0, 1.0, 2.0])
t5_rewards = np.array([0.0, 1.0, 3.0])
t5_shifted = t5_logits - np.max(t5_logits)       # -> [-2.0, -1.0, 0.0]
t5_exp = np.exp(t5_shifted)                      # -> [0.135, 0.368, 1.0]
t5_probs = t5_exp / np.sum(t5_exp)               # -> [0.09, 0.245, 0.665]
t5_expected = float(t5_probs @ t5_rewards)       # -> 2.24
print("shifted logits:", t5_shifted.tolist())    # -> [-2.0, -1.0, 0.0]
print("exp scores:", np.round(t5_exp, 3).tolist())      # -> [0.135, 0.368, 1.0]
print("probabilities:", np.round(t5_probs, 3).tolist()) # -> [0.09, 0.245, 0.665]
print("probability sum:", round(float(np.sum(t5_probs)), 3))  # -> 1.0
print("expected reward:", round(t5_expected, 3))       # -> 2.24

plt.figure(figsize=(4.3, 2.6))
plt.bar(["a0", "a1", "a2"], t5_probs, color="purple")
plt.ylim(0, 1)
plt.title("Toy 5 · softmax policy")
plt.ylabel("probability")
plt.show()
assert np.allclose(np.round(t5_probs, 3), np.array([0.09, 0.245, 0.665]))

▶ What you'll see: action `2` has the largest probability because it has the largest logit.

### ✍️ Toy 6 · Exploration bonuses shrink with counts

The same bonus formula gives a much larger lift to the action with fewer samples.

In [ ]:
import numpy as np

t6_rng = np.random.default_rng(0)
t6_means = np.array([0.8, 0.6])
t6_counts = np.array([16, 2])
t6_time = 32
t6_c = 0.5
t6_bonus = t6_c * np.sqrt(2 * np.log(t6_time) / t6_counts)  # -> [0.329, 0.931]
t6_index = t6_means + t6_bonus                              # -> [1.129, 1.531]
t6_choice = int(np.argmax(t6_index))                        # -> 1
print("means:", t6_means.tolist())                         # -> [0.8, 0.6]
print("counts:", t6_counts.tolist())                       # -> [16, 2]
print("bonuses:", np.round(t6_bonus, 3).tolist())          # -> [0.329, 0.931]
print("indices:", np.round(t6_index, 3).tolist())          # -> [1.129, 1.531]
print("chosen action:", t6_choice)                         # -> 1

plt.figure(figsize=(4.3, 2.6))
plt.bar(["a0", "a1"], t6_means, color="gray", label="mean")
plt.scatter(["a0", "a1"], t6_index, color="crimson", label="mean + bonus", zorder=3)
plt.title("Toy 6 · fewer visits means bigger optimism")
plt.ylabel("score")
plt.legend()
plt.show()
assert t6_choice == 1

▶ What you'll see: action `1` wins despite the lower mean because it has only two visits.

## 🛠️ Setup

In [ ]:
import numpy as np  # Import NumPy for arrays, transition tensors, expectations, and small numerical checks.
import matplotlib.pyplot as plt  # Import Matplotlib for heatmaps, bars, lines, and policy/value diagnostics.
np.random.seed(0)  # Fix the global random seed so every stochastic example is repeatable.

## 🟢 Basics (warm-up)

### Basic 1 — Name states and actions

**Goal.** Build the smallest vocabulary of an MDP, because every later tensor is indexed by state and action names. We build it in 2 steps.

In [ ]:
states_b1 = ["start", "study", "rest"]  # List the possible situations the agent can occupy.
actions_b1 = ["focus", "break"]  # List the decisions available to the agent.
print("states:", states_b1)  # Inspect state labels before turning them into indices.
print("actions:", actions_b1)  # Inspect action labels before building tables.

▶ What you'll see: three named states and two named actions.

In [ ]:
S_b1, A_b1 = len(states_b1), len(actions_b1)  # Count states and actions for tensor shapes.
print("number of states:", S_b1, "number of actions:", A_b1)  # Inspect the dimensions.
assert (S_b1, A_b1) == (3, 2)  # Verify the tiny MDP dimensions.
plt.figure(figsize=(4, 3))
plt.bar(["states", "actions"], [S_b1, A_b1], color=["teal", "orange"])
plt.title("Basic 1: MDP bookkeeping sizes")
plt.ylabel("count")
plt.show()

▶ What you'll see: the state count is larger than the action count in this toy MDP.

👀 Takeaway: an MDP begins by defining the discrete sets that every probability and value table will index.

### Basic 2 — Build one transition distribution

**Goal.** Store $P(s'\mid s,a)$ for one state-action pair, because an action usually leads to a distribution of outcomes rather than one guaranteed next state. We build it in 2 steps.

In [ ]:
next_probs_b2 = np.array([0.10, 0.80, 0.10])  # From start + focus: start, study, rest probabilities.
states_b2 = ["start", "study", "rest"]  # Keep labels beside the probability vector.
print("next-state probabilities:", next_probs_b2)  # Inspect the transition distribution.
print("sum:", next_probs_b2.sum())  # Check that probabilities form a distribution.
assert np.isclose(next_probs_b2.sum(), 1.0)  # Probabilities must sum to one.

▶ What you'll see: most probability mass goes to `study`, and the row sums to 1.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.bar(states_b2, next_probs_b2, color="seagreen")
plt.title("Basic 2: P(s' | start, focus)")
plt.ylabel("probability")
plt.ylim(0, 1)
plt.show()

▶ What you'll see: `study` is the tallest bar because focusing usually moves the agent there.

👀 Takeaway: a transition row is a probability distribution over possible next states.

### Basic 3 — Attach rewards to realized transitions

**Goal.** Pair next-state probabilities with rewards, because expected consequence multiplies how likely an outcome is by how valuable it is. We build it in 2 steps.

In [ ]:
probs_b3 = np.array([0.10, 0.80, 0.10])  # Transition probabilities for one action.
rewards_b3 = np.array([-0.2, 2.0, 0.5])  # Reward received if the transition lands in each next state.
weighted_b3 = probs_b3 * rewards_b3  # Probability-weighted reward contributions.
print("weighted reward pieces:", np.round(weighted_b3, 3))  # Inspect each outcome's expected contribution.

▶ What you'll see: the high-probability `study` outcome dominates the expected reward.

In [ ]:
expected_reward_b3 = float(np.sum(weighted_b3))  # Sum outcome contributions into expected immediate reward.
print("expected immediate reward:", round(expected_reward_b3, 3))
assert round(expected_reward_b3, 3) == 1.630
plt.figure(figsize=(4.5, 3))
plt.bar(["start", "study", "rest"], weighted_b3, color="teal")
plt.title("Basic 3: probability × reward")
plt.ylabel("expected contribution")
plt.show()

▶ What you'll see: the expected immediate reward is `1.630` after averaging over stochastic outcomes.

👀 Takeaway: rewards are consequences of transitions, and expected reward is probability-weighted reward.

### Basic 4 — Compute a discounted return

**Goal.** Collapse a short reward stream into one return, because value learning needs one target number for future consequence. We build it in 2 steps.

In [ ]:
rewards_b4 = np.array([1.0, 0.0, 2.0])  # Three rewards collected over time.
gamma_b4 = 0.9  # Future rewards retain 90% of their value per step.
discounts_b4 = gamma_b4 ** np.arange(len(rewards_b4))  # Compute 1, gamma, gamma squared.
print("discounts:", np.round(discounts_b4, 3))  # Inspect discount weights.

▶ What you'll see: the discount weights are `1.0`, `0.9`, and `0.81`.

In [ ]:
return_b4 = float(np.sum(discounts_b4 * rewards_b4))  # Sum discounted reward pieces.
print("discounted return:", round(return_b4, 3))
assert round(return_b4, 3) == 2.620
plt.figure(figsize=(4, 3))
plt.bar(["t0", "t1", "t2"], discounts_b4 * rewards_b4, color="purple")
plt.title("Basic 4: return contributions")
plt.ylabel("γ^t r_t")
plt.show()

▶ What you'll see: the final reward contributes `1.62` instead of `2.0` because it is two steps away.

👀 Takeaway: return is reward plus discounted future reward, not just the immediate reward.

### Basic 5 — Compare different discount factors

**Goal.** See how $\gamma$ changes the same reward stream, because discounting controls how farsighted the agent is. We build it in 2 steps.

In [ ]:
rewards_b5 = np.array([0.0, 0.0, 5.0])  # A payoff that arrives only after waiting.
gammas_b5 = np.array([0.2, 0.5, 0.9, 0.99])  # Myopic to farsighted discounts.
returns_b5 = np.array([np.sum((g_b5 ** np.arange(len(rewards_b5))) * rewards_b5) for g_b5 in gammas_b5])  # Compute each return.
print("returns:", np.round(returns_b5, 3))  # Inspect how much the delayed payoff is worth.

▶ What you'll see: high gamma values preserve much more of the delayed reward.

In [ ]:
assert round(float(returns_b5[2]), 3) == 4.050  # With gamma 0.9, reward two steps away counts as 0.81*5.
plt.figure(figsize=(4.5, 3))
plt.plot(gammas_b5, returns_b5, marker="o", color="crimson")
plt.title("Basic 5: discount factor changes planning horizon")
plt.xlabel("gamma")
plt.ylabel("return")
plt.show()

▶ What you'll see: the curve rises as gamma gets closer to 1.

👀 Takeaway: larger discount factors make delayed consequences matter more.

### Basic 6 — Convert logits to a softmax policy

**Goal.** Turn unconstrained action scores into probabilities, because policies must sum to one and stay nonnegative. We build it in 2 steps.

In [ ]:
logits_b6 = np.array([1.0, 0.0])  # Preference scores for focus and break.
exp_b6 = np.exp(logits_b6 - np.max(logits_b6))  # Stabilize before exponentiating.
probs_b6 = exp_b6 / exp_b6.sum()  # Normalize exponentials into probabilities.
print("policy probabilities:", np.round(probs_b6, 3))
assert np.allclose(np.round(probs_b6, 3), [0.731, 0.269])

▶ What you'll see: the higher logit receives about 73.1% probability.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["focus", "break"], probs_b6, color="darkorange")
plt.title("Basic 6: softmax policy")
plt.ylabel("π(a|s)")
plt.ylim(0, 1)
plt.show()

▶ What you'll see: softmax favors `focus` but keeps nonzero probability on `break`.

👀 Takeaway: softmax converts arbitrary scores into a valid stochastic policy.

### Basic 7 — Expected reward under a policy

**Goal.** Average action rewards using policy probabilities, because stochastic policies choose actions with different frequencies. We build it in 2 steps.

In [ ]:
policy_b7 = np.array([0.731, 0.269])  # Probability of focus and break.
action_rewards_b7 = np.array([2.0, 0.0])  # Immediate rewards for the two actions.
pieces_b7 = policy_b7 * action_rewards_b7  # Each action's contribution to expected reward.
print("policy-weighted reward pieces:", np.round(pieces_b7, 3))

▶ What you'll see: the high-reward action contributes only in proportion to its probability.

In [ ]:
expected_b7 = float(np.sum(pieces_b7))  # Sum policy-weighted rewards.
print("expected reward:", round(expected_b7, 3))
assert round(expected_b7, 3) == 1.462
plt.figure(figsize=(4, 3))
plt.bar(["focus", "break"], pieces_b7, color="seagreen")
plt.title("Basic 7: policy-weighted reward")
plt.ylabel("π(a) × r(a)")
plt.show()

▶ What you'll see: expected reward is below 2 because the policy sometimes takes the zero-reward action.

👀 Takeaway: policy evaluation is expectation arithmetic over actions and outcomes.

### Basic 8 — Make a one-step bootstrap target

**Goal.** Combine observed reward with an estimated next value, because temporal-difference learning bootstraps from the current value table. We build it in 2 steps.

In [ ]:
r_b8 = 1.0  # Reward observed on the transition.
gamma_b8 = 0.9  # Discount on the next state's value.
next_v_b8 = 0.8  # Current estimate for the next state.
target_b8 = r_b8 + gamma_b8 * next_v_b8  # One-step TD target.
print("target:", round(target_b8, 3))
assert round(target_b8, 3) == 1.720

▶ What you'll see: the bootstrap target is `1.720`.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["reward", "γ next V", "target"], [r_b8, gamma_b8 * next_v_b8, target_b8], color=["gray", "teal", "purple"])
plt.title("Basic 8: TD target pieces")
plt.ylabel("value")
plt.show()

▶ What you'll see: the target is immediate reward plus discounted next-state estimate.

👀 Takeaway: bootstrapping uses a learned estimate as part of the target, rather than waiting for the full return.

### Basic 9 — Move an estimate toward a target

**Goal.** Apply $Q_{new}=Q_{old}+\alpha(y-Q_{old})$, because value tables should update gradually rather than jump to one noisy sample. We build it in 2 steps.

In [ ]:
q_old_b9 = 0.4  # Current Q estimate.
target_b9 = 1.72  # Bootstrap target from the previous worked calculation.
alpha_b9 = 0.5  # Move halfway toward the target.
td_error_b9 = target_b9 - q_old_b9  # Difference between target and current estimate.
print("TD error:", round(td_error_b9, 3))

▶ What you'll see: the estimate is `1.32` below the target.

In [ ]:
q_new_b9 = q_old_b9 + alpha_b9 * td_error_b9  # Incremental TD update.
print("new Q:", round(q_new_b9, 3))
assert round(q_new_b9, 3) == 1.060
plt.figure(figsize=(4, 3))
plt.bar(["old", "target", "new"], [q_old_b9, target_b9, q_new_b9], color=["gray", "black", "teal"])
plt.title("Basic 9: partial move toward target")
plt.ylabel("Q value")
plt.show()

▶ What you'll see: the new value lands halfway between old value and target.

👀 Takeaway: the learning rate controls how strongly one target changes the estimate.

### Basic 10 — Add an exploration bonus

**Goal.** Compute a UCB-style index, because uncertain actions can deserve temporary optimism even when their current mean is modest. We build it in 2 steps.

In [ ]:
mean_b10 = 0.55  # Current empirical mean reward for an action.
time_b10 = 20  # Total decision count.
count_b10 = 5  # Number of times this action was sampled.
bonus_b10 = np.sqrt(2 * np.log(time_b10) / count_b10)  # UCB uncertainty bonus with c=1.
print("bonus:", round(bonus_b10, 3))

▶ What you'll see: the bonus is larger when count is small relative to time.

In [ ]:
ucb_b10 = mean_b10 + bonus_b10  # Optimistic action index.
print("UCB score:", round(ucb_b10, 3))
assert round(ucb_b10, 3) == 1.645
plt.figure(figsize=(4, 3))
plt.bar(["mean", "bonus", "UCB"], [mean_b10, bonus_b10, ucb_b10], color=["gray", "orange", "crimson"])
plt.title("Basic 10: optimism under uncertainty")
plt.ylabel("index")
plt.show()

▶ What you'll see: the index is much larger than the observed mean because uncertainty is valuable.

👀 Takeaway: exploration bonuses trade short-term greed for information that can improve future decisions.

## 🟡 Easy

### Easy 1 — Build a full transition and reward model

**Goal.** Create full $P$ and $R$ tensors for a toy MDP, because Bellman backups need a model for every state-action pair. We build it in 3 steps.

In [ ]:
states_e1 = ["start", "study", "rest"]  # State labels for readable output.
actions_e1 = ["focus", "break"]  # Action labels for readable output.
P_e1 = np.array([[[0.10, 0.80, 0.10], [0.20, 0.20, 0.60]],
                 [[0.00, 0.70, 0.30], [0.10, 0.20, 0.70]],
                 [[0.10, 0.60, 0.30], [0.20, 0.10, 0.70]]])  # P[s,a,s'].
print("transition tensor shape:", P_e1.shape)

▶ What you'll see: the transition model has one probability row for each state-action pair.

In [ ]:
R_e1 = np.array([[[-0.2, 2.0, 0.5], [0.0, 0.4, 0.8]],
                 [[0.0, 1.0, 0.2], [0.1, 0.3, 1.0]],
                 [[0.0, 1.5, 0.6], [0.2, 0.1, 0.7]]])  # R[s,a,s'].
print("all transition rows sum to one:", np.allclose(P_e1.sum(axis=2), 1.0))
assert np.allclose(P_e1.sum(axis=2), 1.0)

In [ ]:
plt.figure(figsize=(5, 3))
plt.imshow(P_e1.reshape(-1, 3), cmap="viridis", aspect="auto")
plt.colorbar(label="probability")
plt.yticks(range(6), [f"{s}-{a}" for s in states_e1 for a in actions_e1])
plt.xticks(range(3), states_e1)
plt.title("Easy 1: transition rows")
plt.show()

▶ What you'll see: each row is a next-state distribution for one state-action pair.

👀 Takeaway: a tabular MDP model is just probability and reward tensors with carefully checked shapes.

### Easy 2 — Evaluate one Bellman state backup

**Goal.** Compute $V^\pi(s)$ for one state from $P$, $R$, $\pi$, and old values, because Bellman evaluation is nested expectation. We build it in 3 steps.

In [ ]:
P_e2 = np.array([[[0.10, 0.80, 0.10], [0.20, 0.20, 0.60]],
                 [[0.00, 0.70, 0.30], [0.10, 0.20, 0.70]],
                 [[0.10, 0.60, 0.30], [0.20, 0.10, 0.70]]])
R_e2 = np.array([[[-0.2, 2.0, 0.5], [0.0, 0.4, 0.8]],
                 [[0.0, 1.0, 0.2], [0.1, 0.3, 1.0]],
                 [[0.0, 1.5, 0.6], [0.2, 0.1, 0.7]]])
V_e2 = np.array([0.0, 1.2, 0.7])  # Current value estimates.
print("old V:", V_e2)

▶ What you'll see: the backup will reuse the current values of possible next states.

In [ ]:
pi_e2 = np.array([[0.75, 0.25], [0.65, 0.35], [0.55, 0.45]])  # Policy probabilities.
gamma_e2 = 0.9  # Discount factor.
s_e2 = 0  # Evaluate the start state.
q_like_e2 = np.sum(P_e2[s_e2] * (R_e2[s_e2] + gamma_e2 * V_e2), axis=1)  # Expected consequence for each action.
print("action consequences:", np.round(q_like_e2, 3))

In [ ]:
backup_e2 = float(np.dot(pi_e2[s_e2], q_like_e2))  # Policy-weighted Bellman backup.
print("V backup for start:", round(backup_e2, 3))
assert round(backup_e2, 3) == 2.206
plt.figure(figsize=(4, 3))
plt.bar(["focus", "break"], q_like_e2, color="teal")
plt.title("Easy 2: action terms inside V backup")
plt.ylabel("expected r + γV")
plt.show()

▶ What you'll see: the policy-weighted backup for `start` is about `2.206`.

👀 Takeaway: Bellman evaluation turns a state value into an average over policy actions and stochastic next states.

### Easy 3 — Run iterative policy evaluation

**Goal.** Repeatedly apply Bellman backups, because values are fixed points of the backup equation rather than one-shot quantities. We build it in 3 steps.

In [ ]:
P_e3 = np.array([[[0.10, 0.80, 0.10], [0.20, 0.20, 0.60]],
                 [[0.00, 0.70, 0.30], [0.10, 0.20, 0.70]],
                 [[0.10, 0.60, 0.30], [0.20, 0.10, 0.70]]])
R_e3 = np.array([[[-0.2, 2.0, 0.5], [0.0, 0.4, 0.8]],
                 [[0.0, 1.0, 0.2], [0.1, 0.3, 1.0]],
                 [[0.0, 1.5, 0.6], [0.2, 0.1, 0.7]]])
pi_e3 = np.array([[0.75, 0.25], [0.65, 0.35], [0.55, 0.45]])
V_e3 = np.zeros(3)  # Start with no long-run knowledge.
print("initial V:", V_e3)

▶ What you'll see: value iteration begins from zeros.

In [ ]:
gamma_e3 = 0.9
history_e3 = []
for it_e3 in range(25):
    Q_e3 = np.sum(P_e3 * (R_e3 + gamma_e3 * V_e3[None, None, :]), axis=2)  # One-step action consequences.
    V_e3 = np.sum(pi_e3 * Q_e3, axis=1)  # Average by policy probabilities.
    history_e3.append(V_e3.copy())
print("value after 25 sweeps:", np.round(V_e3, 3))
assert np.all(V_e3 > 0)

In [ ]:
history_e3 = np.array(history_e3)
plt.figure(figsize=(5, 3))
for idx_e3, name_e3 in enumerate(["start", "study", "rest"]):
    plt.plot(history_e3[:, idx_e3], label=name_e3)
plt.title("Easy 3: policy evaluation convergence")
plt.xlabel("sweep")
plt.ylabel("V")
plt.legend()
plt.show()

▶ What you'll see: the value curves rise and begin to flatten as repeated backups approach a fixed point.

👀 Takeaway: iterative policy evaluation is repeated expectation backup until values stabilize.

### Easy 4 — Compute Q-values and choose greedy actions

**Goal.** Convert state values into action values and select the best action per state, because policy improvement compares actions directly. We build it in 3 steps.

In [ ]:
P_e4 = np.array([[[0.10, 0.80, 0.10], [0.20, 0.20, 0.60]],
                 [[0.00, 0.70, 0.30], [0.10, 0.20, 0.70]],
                 [[0.10, 0.60, 0.30], [0.20, 0.10, 0.70]]])
R_e4 = np.array([[[-0.2, 2.0, 0.5], [0.0, 0.4, 0.8]],
                 [[0.0, 1.0, 0.2], [0.1, 0.3, 1.0]],
                 [[0.0, 1.5, 0.6], [0.2, 0.1, 0.7]]])
V_e4 = np.array([12.0, 13.0, 11.5])  # A plausible long-run value estimate.
print("candidate V:", V_e4)

▶ What you'll see: each next state has a current long-run value estimate.

In [ ]:
gamma_e4 = 0.9
Q_e4 = np.sum(P_e4 * (R_e4 + gamma_e4 * V_e4[None, None, :]), axis=2)  # Q[s,a] from one Bellman lookahead.
greedy_e4 = np.argmax(Q_e4, axis=1)  # Best action index in each state.
print("Q table:\n", np.round(Q_e4, 3))
print("greedy actions:", greedy_e4)
assert Q_e4.shape == (3, 2)

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.imshow(Q_e4, cmap="viridis", aspect="auto")
plt.colorbar(label="Q value")
plt.xticks([0, 1], ["focus", "break"])
plt.yticks([0, 1, 2], ["start", "study", "rest"])
plt.title("Easy 4: action-value table")
plt.show()

▶ What you'll see: the brighter action in each row is the greedy action for that state.

👀 Takeaway: $Q(s,a)$ exposes action choices that a scalar state value hides.

### Easy 5 — Simulate a tiny trajectory

**Goal.** Sample transitions from the MDP and compute the realized discounted return, because expected values summarize many possible trajectories. We build it in 3 steps.

In [ ]:
P_e5 = np.array([[[0.10, 0.80, 0.10], [0.20, 0.20, 0.60]],
                 [[0.00, 0.70, 0.30], [0.10, 0.20, 0.70]],
                 [[0.10, 0.60, 0.30], [0.20, 0.10, 0.70]]])
R_e5 = np.array([[[-0.2, 2.0, 0.5], [0.0, 0.4, 0.8]],
                 [[0.0, 1.0, 0.2], [0.1, 0.3, 1.0]],
                 [[0.0, 1.5, 0.6], [0.2, 0.1, 0.7]]])
pi_e5 = np.array([[0.75, 0.25], [0.65, 0.35], [0.55, 0.45]])
rng_e5 = np.random.default_rng(5)
print("start state index:", 0)

▶ What you'll see: the simulation begins at state `0`.

In [ ]:
s_e5 = 0
states_seen_e5 = [s_e5]
rewards_e5 = []
for t_e5 in range(6):
    a_e5 = rng_e5.choice(2, p=pi_e5[s_e5])  # Sample action from policy.
    s_next_e5 = rng_e5.choice(3, p=P_e5[s_e5, a_e5])  # Sample next state from transition row.
    rewards_e5.append(R_e5[s_e5, a_e5, s_next_e5])  # Record realized reward.
    s_e5 = s_next_e5
    states_seen_e5.append(s_e5)
print("states seen:", states_seen_e5)
print("rewards:", np.round(rewards_e5, 3))

In [ ]:
gamma_e5 = 0.9
G_e5 = float(np.sum((gamma_e5 ** np.arange(len(rewards_e5))) * np.array(rewards_e5)))
print("realized discounted return:", round(G_e5, 3))
assert len(rewards_e5) == 6
plt.figure(figsize=(5, 3))
plt.step(range(len(states_seen_e5)), states_seen_e5, where="post", color="purple")
plt.title("Easy 5: sampled state trajectory")
plt.xlabel("time")
plt.ylabel("state index")
plt.show()

▶ What you'll see: one random path through states, with a single realized return for that path.

👀 Takeaway: values are expectations, while a trajectory is one sampled realization from the MDP.

## 🔴 Advanced

### Advanced 1 — Value iteration for the optimal policy

**Goal.** Replace policy averaging with max over actions, because optimal control asks for the best action rather than the value of a fixed policy. We build it in 4 steps.

In [ ]:
P_a1 = np.array([[[0.10, 0.80, 0.10], [0.20, 0.20, 0.60]],
                 [[0.00, 0.70, 0.30], [0.10, 0.20, 0.70]],
                 [[0.10, 0.60, 0.30], [0.20, 0.10, 0.70]]])
R_a1 = np.array([[[-0.2, 2.0, 0.5], [0.0, 0.4, 0.8]],
                 [[0.0, 1.0, 0.2], [0.1, 0.3, 1.0]],
                 [[0.0, 1.5, 0.6], [0.2, 0.1, 0.7]]])
V_a1 = np.zeros(3)
gamma_a1 = 0.9
print("initial optimal-value guess:", V_a1)

▶ What you'll see: optimal value iteration starts from zeros.

In [ ]:
history_a1 = []
for sweep_a1 in range(40):
    Q_a1 = np.sum(P_a1 * (R_a1 + gamma_a1 * V_a1[None, None, :]), axis=2)  # Look ahead for all actions.
    V_a1 = np.max(Q_a1, axis=1)  # Optimal Bellman backup chooses the best action.
    history_a1.append(V_a1.copy())
print("optimal V after sweeps:", np.round(V_a1, 3))
assert np.all(V_a1 > 0)

In [ ]:
Q_final_a1 = np.sum(P_a1 * (R_a1 + gamma_a1 * V_a1[None, None, :]), axis=2)
policy_a1 = np.argmax(Q_final_a1, axis=1)
print("greedy optimal action indices:", policy_a1)
assert policy_a1.shape == (3,)

In [ ]:
history_a1 = np.array(history_a1)
plt.figure(figsize=(5, 3))
for i_a1, name_a1 in enumerate(["start", "study", "rest"]):
    plt.plot(history_a1[:, i_a1], label=name_a1)
plt.title("Advanced 1: value iteration")
plt.xlabel("sweep")
plt.ylabel("V*")
plt.legend()
plt.show()

▶ What you'll see: values increase and flatten as the max-backup approaches an optimal fixed point.

👀 Takeaway: value iteration is Bellman backup plus maximization, yielding both optimal values and a greedy policy.

### Advanced 2 — Q-learning from sampled transitions

**Goal.** Learn action values without using the full transition model in the update, because model-free RL bootstraps from samples. We build it in 4 steps.

In [ ]:
P_a2 = np.array([[[0.10, 0.80, 0.10], [0.20, 0.20, 0.60]],
                 [[0.00, 0.70, 0.30], [0.10, 0.20, 0.70]],
                 [[0.10, 0.60, 0.30], [0.20, 0.10, 0.70]]])
R_a2 = np.array([[[-0.2, 2.0, 0.5], [0.0, 0.4, 0.8]],
                 [[0.0, 1.0, 0.2], [0.1, 0.3, 1.0]],
                 [[0.0, 1.5, 0.6], [0.2, 0.1, 0.7]]])
Q_a2 = np.zeros((3, 2))
rng_a2 = np.random.default_rng(2)
print("initial Q shape:", Q_a2.shape)

▶ What you'll see: Q-learning stores one estimate per state-action pair.

In [ ]:
alpha_a2, gamma_a2, eps_a2 = 0.25, 0.9, 0.2
s_a2 = 0
for step_a2 in range(300):
    if rng_a2.random() < eps_a2:
        a_a2 = rng_a2.integers(2)  # Explore.
    else:
        a_a2 = int(np.argmax(Q_a2[s_a2]))  # Exploit current Q.
    sp_a2 = rng_a2.choice(3, p=P_a2[s_a2, a_a2])
    r_a2 = R_a2[s_a2, a_a2, sp_a2]
    target_a2 = r_a2 + gamma_a2 * np.max(Q_a2[sp_a2])
    Q_a2[s_a2, a_a2] += alpha_a2 * (target_a2 - Q_a2[s_a2, a_a2])
    s_a2 = sp_a2
print("learned Q:\n", np.round(Q_a2, 2))

In [ ]:
learned_policy_a2 = np.argmax(Q_a2, axis=1)
print("learned greedy actions:", learned_policy_a2)
assert Q_a2.shape == (3, 2)

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.imshow(Q_a2, cmap="viridis", aspect="auto")
plt.colorbar(label="Q")
plt.xticks([0, 1], ["focus", "break"])
plt.yticks([0, 1, 2], ["start", "study", "rest"])
plt.title("Advanced 2: sampled Q-learning table")
plt.show()

▶ What you'll see: sampled bootstrapping fills a Q table even though each update observed only one transition.

👀 Takeaway: Q-learning replaces model expectation with sample experience and a max next-action bootstrap.

### Advanced 3 — Monte Carlo return versus TD target

**Goal.** Compare complete-return targets with one-step TD targets, because target choice controls the bias-variance tradeoff in RL. We build it in 3 steps.

In [ ]:
rewards_a3 = np.array([1.0, 0.0, 2.0, -0.5])  # A short realized episode tail.
gamma_a3 = 0.9
mc_return_a3 = float(np.sum((gamma_a3 ** np.arange(len(rewards_a3))) * rewards_a3))
next_v_a3 = 0.8
td_target_a3 = rewards_a3[0] + gamma_a3 * next_v_a3
print("Monte Carlo return:", round(mc_return_a3, 3))
print("one-step TD target:", round(td_target_a3, 3))

▶ What you'll see: the full return and one-step target are different targets for the same first state.

In [ ]:
assert round(td_target_a3, 3) == 1.720
assert round(mc_return_a3, 3) == 2.256
labels_a3 = ["MC full return", "TD one-step"]
values_a3 = [mc_return_a3, td_target_a3]
print("target gap:", round(mc_return_a3 - td_target_a3, 3))

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.bar(labels_a3, values_a3, color=["purple", "teal"])
plt.title("Advanced 3: target choice")
plt.ylabel("target value")
plt.xticks(rotation=15)
plt.show()

▶ What you'll see: Monte Carlo includes all realized rewards, while TD uses one reward plus an estimate.

👀 Takeaway: Monte Carlo is less biased but noisier; TD is more biased but can learn earlier from partial experience.

### Advanced 4 — Policy-gradient intuition with advantages

**Goal.** Weight log-probability changes by advantage, because policy updates should reinforce actions that did better than the state's baseline and discourage actions that did worse. We build it in 4 steps.

In [ ]:
logits_a4 = np.array([0.4, -0.2])  # Current action logits.
exp_a4 = np.exp(logits_a4 - np.max(logits_a4))
pi_a4 = exp_a4 / exp_a4.sum()
q_values_a4 = np.array([2.4, 1.0])  # Estimated action values.
V_a4 = float(np.dot(pi_a4, q_values_a4))  # State value under the current policy.
print("policy:", np.round(pi_a4, 3), "state value:", round(V_a4, 3))

▶ What you'll see: the state value is the policy-weighted average of action values.

In [ ]:
advantages_a4 = q_values_a4 - V_a4  # A(s,a)=Q(s,a)-V(s).
print("advantages:", np.round(advantages_a4, 3))
assert np.isclose(np.dot(pi_a4, advantages_a4), 0.0)

In [ ]:
chosen_a4 = 0
eta_a4 = 0.4
grad_logits_a4 = -pi_a4.copy()
grad_logits_a4[chosen_a4] += 1.0  # Gradient of log pi(chosen action) wrt logits.
new_logits_a4 = logits_a4 + eta_a4 * advantages_a4[chosen_a4] * grad_logits_a4
new_pi_a4 = np.exp(new_logits_a4 - np.max(new_logits_a4)); new_pi_a4 = new_pi_a4 / new_pi_a4.sum()
print("new policy after reinforcing action 0:", np.round(new_pi_a4, 3))
assert new_pi_a4[0] > pi_a4[0]

In [ ]:
plt.figure(figsize=(4, 3))
x_a4 = np.arange(2)
plt.bar(x_a4 - 0.18, pi_a4, width=0.36, label="before", color="gray")
plt.bar(x_a4 + 0.18, new_pi_a4, width=0.36, label="after", color="teal")
plt.xticks(x_a4, ["focus", "break"])
plt.title("Advanced 4: advantage shifts policy mass")
plt.ylabel("probability")
plt.legend()
plt.show()

▶ What you'll see: probability moves toward the positive-advantage action.

👀 Takeaway: advantages tell a policy update whether an action was better or worse than expected for that state.

### Advanced 5 — Policy support in offline evaluation

**Goal.** Diagnose when an offline policy estimate is unreliable, because actions rarely sampled by the behavior policy create high-variance importance weights. We build it in 4 steps.

In [ ]:
behavior_a5 = np.array([0.95, 0.05])  # Logged data policy: almost always action 0.
target_a5 = np.array([0.50, 0.50])  # Policy we want to evaluate offline.
rewards_logged_a5 = np.array([1.0, 4.0])  # Observed reward if each action is sampled.
weights_a5 = target_a5 / behavior_a5  # Importance ratios per action.
print("importance weights:", np.round(weights_a5, 3))

▶ What you'll see: action 1 gets a huge weight because it was rarely sampled by the behavior policy.

In [ ]:
expected_logged_a5 = float(np.dot(behavior_a5, rewards_logged_a5))
expected_target_a5 = float(np.dot(target_a5, rewards_logged_a5))
weighted_estimate_a5 = float(np.dot(behavior_a5, weights_a5 * rewards_logged_a5))
print("behavior expected reward:", round(expected_logged_a5, 3))
print("target expected reward via weights:", round(weighted_estimate_a5, 3))
assert round(weighted_estimate_a5, 3) == round(expected_target_a5, 3)

In [ ]:
n_samples_a5 = 200
expected_action1_count_a5 = n_samples_a5 * behavior_a5[1]
print("expected logged samples of rare action:", round(expected_action1_count_a5, 1))
assert round(expected_action1_count_a5, 1) == 10.0

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(["action 0", "action 1"], weights_a5, color=["teal", "crimson"])
plt.title("Advanced 5: rare actions create large weights")
plt.ylabel("target π / behavior μ")
plt.show()

▶ What you'll see: the rarely logged action has a large importance weight, making offline estimates fragile.

👀 Takeaway: off-policy evaluation needs behavior-policy support; if an action was almost never sampled, its estimate can be dominated by a few examples.